In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-02 23:08:10.677478: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-02 23:08:13.183484: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "mode": 'local',
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

2023-07-02 23:08:18,158 [DEBUG] [Rain] Rain is initialized
2023-07-02 23:08:18,176 [DEBUG] [Provisioner] Creating coordinator
2023-07-02 23:08:18,178 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-02 23:08:18,221 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-02 23:08:18,266 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized


In [9]:
# model = rain.train_centralized_async()

In [10]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [11]:
model = rain.train_centralized_sync()

2023-07-02 23:08:18,871 [DEBUG] [Rain] Creating workers
2023-07-02 23:08:18,886 [INFO] [Provisioner] provisioner is serving
2023-07-02 23:08:18,888 [DEBUG] [Provisioner] Starting coordinator
2023-07-02 23:08:18,891 [INFO] [Coordinator] coordinator is serving
2023-07-02 23:08:18,893 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-02 23:08:18,909 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-02 23:08:18,925 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-02 23:08:18,927 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-02 23:08:18,932 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-02 23:08:18,937 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-02 23:08:18,941 [INFO] [Worker_50153] Worker is running on port: 50153
2023-07-02 23:08:18,959 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 6s 25ms/step - loss: 0.7017 - accuracy: 0.7771
Epoch 2/2
157/157 [==============================] - 6s 25ms/step - loss: 0.7089 - accuracy: 0.7753
Epoch 2/2
157/157 [==============================] - 6s 25ms/step - loss: 0.7032 - accuracy: 0.7786
Epoch 2/2
157/157 [==============================] - 3s 21ms/step - loss: 0.3012 - accuracy: 0.9097
sending data to coordinator


2023-07-02 23:10:46,113 [DEBUG] [DividerAmbassador] divider received: Executed! for worker3
2023-07-02 23:10:46,143 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/3_1_trained.pkl from worker3
2023-07-02 23:10:46,305 [DEBUG] [DividerAmbassador] divider received: Executed! for worker2
2023-07-02 23:10:46,308 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/2_1_trained.pkl from worker2
2023-07-02 23:10:46,325 [DEBUG] [DividerAmbassador] divider received: Executed! for worker1
2023-07-02 23:10:46,330 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/1_1_trained.pkl from worker1
2023-07-02 23:10:47,270 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/3_1_trained.pkl in divider
2023-07-02 23:10:47,710 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/2_1_trained.pkl in divider
2023-07-02 23:10:47,723 [DEBUG] [DividerAmbassador] Downloaded ../../.

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 6s 25ms/step - loss: 0.2502 - accuracy: 0.9255
Epoch 2/2
Epoch 2/2
157/157 [==============================] - 4s 22ms/step - loss: 0.2037 - accuracy: 0.9390
sending data to coordinator
157/157 [==============================] - 4s 23ms/step - loss: 0.1973 - accuracy: 0.9406


2023-07-02 23:11:15,056 [DEBUG] [DividerAmbassador] divider received: Executed! for worker1
2023-07-02 23:11:15,061 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/1_2_trained.pkl from worker1
2023-07-02 23:11:15,338 [DEBUG] [DividerAmbassador] divider received: Executed! for worker2
2023-07-02 23:11:15,341 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/2_2_trained.pkl from worker2
2023-07-02 23:11:15,427 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/1_2_trained.pkl in divider
2023-07-02 23:11:16,035 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/2_2_trained.pkl in divider


...

2023-07-02 23:11:29.927553: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 62720000 exceeds 10% of free system memory.


Epoch 1/2
Epoch 1/2
157/157 [==============================] - 3s 21ms/step - loss: 0.1534 - accuracy: 0.9541
sending data to coordinator


2023-07-02 23:11:40,540 [DEBUG] [DividerAmbassador] divider received: Executed! for worker3
2023-07-02 23:11:40,542 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/3_3_trained.pkl from worker3
2023-07-02 23:11:40,910 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/3_3_trained.pkl in divider
2023-07-02 23:11:41,074 [DEBUG] [DividerAmbassador] divider received: Executed! for worker2
2023-07-02 23:11:41,077 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/2_3_trained.pkl from worker2
2023-07-02 23:11:41,505 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/2_3_trained.pkl in divider


sending data to coordinator


2023-07-02 23:11:42,962 [DEBUG] [DividerAmbassador] divider received: Executed! for worker1
2023-07-02 23:11:42,963 [DEBUG] [DividerAmbassador] divider begins downloading ../../../Divider/divider/data/1_3_trained.pkl from worker1
2023-07-02 23:11:43,510 [DEBUG] [DividerAmbassador] Downloaded ../../../Divider/divider/data/1_3_trained.pkl in divider
2023-07-02 23:11:43,630 [DEBUG] [Divider] Iteration 3/3 complete.
2023-07-02 23:11:43,633 [INFO] [Provisioner] provisioner stopped serving
2023-07-02 23:11:43,645 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-02 23:11:43,650 [DEBUG] [Divider] Divider stopped serving


In [12]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 6ms/step - loss: 0.0957 - accuracy: 0.9711

Test accuracy: 97.1%
